# scaffold-tuner

This notebook evaluates scaffold-tuner on the same fixed parent set used by the other benchmark methods.

Generation uses only the public high-level API shown in the scaffold-tuner usage example:

```python
propose_structures(...)
```

No internal scaffold-tuner functions, fragment-template tables, R-group utilities, or private variables are accessed directly.

This is intentional: one advantage of scaffold-tuner is that a user can request a directed molecular edit with a single function call.


## Preparation


### Import library


In [1]:
import sys

import pandas as pd
from tqdm import tqdm

from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

sys.path.append("..")

from configs.benchmark_config import N_CANDIDATES_PER_PARENT, RANDOM_SEED
from utils.descriptors import calc_descriptors
from utils.parents import prepare_parent
from utils.records import append_candidate_record

!pip install git+https://github.com/tesaki2019/scaffold-tuner.git
from scaffold_tuner.scaffold_intervention import propose_structures

  Cloning https://github.com/tesaki2019/scaffold-tuner.git to /private/var/folders/t4/pnbdyxln7vd9sy0gbp8pz2dm0000gn/T/pip-req-build-95u3n2s6
  Running command git clone --filter=blob:none --quiet https://github.com/tesaki2019/scaffold-tuner.git /private/var/folders/t4/pnbdyxln7vd9sy0gbp8pz2dm0000gn/T/pip-req-build-95u3n2s6
  Resolved https://github.com/tesaki2019/scaffold-tuner.git to commit a3f087d908176ebb2261c993f1484a8a04925ac0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scaffold-tuner: filename=scaffold_tuner-1.0.4-py3-none-any.whl size=11775 sha256=a590f086ef0bf77376e024edf63ddb285fa9e1415d7281a1a88f9521a63548a9
  Stored in directory: /private/var/folders/t4/pnbdyxln7vd9sy0gbp8pz2dm0000gn/T/pip-ephem-wheel-cache-uumpynm0/wheels/eb/fe/99/d0f21ae19b346109ce65262ae891fffc410ced8ced2e7de42b
Successfully built scaffold-tuner
  Attempting uninstall: scaffold-tuner
    Found e

### Import parent compounds


In [2]:
df = pd.read_csv("../data/chembl_1000_parents.csv")

print(f"Parent compounds: {len(df)}")
df.head()


Parent compounds: 1000


,chembl_id,smiles,mw,hbd,hba,logp,rotb,ar
0,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,367.449,2,4,4.49360,9,2
1,CHEMBL91137,CCCc1nnc([S+]([O-])Cc2ncc(C)c(OC)c2C)o1,309.391,0,6,2.35034,6,2
2,CHEMBL441229,C=C1C(O)C(O)C(O)C(O)C1O,176.168,5,5,-2.63930,0,0
3,CHEMBL1068,NC(=O)N1c2ccccc2CC(=O)c2ccccc21,252.273,1,2,2.64220,0,2
4,CHEMBL105373,COc1cc(-n2sc3ncccc3c2=O)cc(OC)c1OC,318.354,0,6,2.47300,4,3


## Compounds generation: scaffold-tuner

The eight directed editing modes are evaluated separately.

For each parent and mode, candidate structures are generated directly with `propose_structures()` requests the strict single-descriptor editing behavior of scaffold-tuner, while `max_candidates` limits the returned proposals per parent and mode.

The benchmark does not reproduce scaffold-tuner's internal chemistry logic. It simply calls the public API as an end user would.


In [3]:
ALL_MODES = [
    "add_hbd",
    "remove_hbd",
    "add_hba",
    "remove_hba",
    "add_ar",
    "remove_ar",
    "add_rb",
    "remove_rb",
]


In [4]:
records = []

for mode in ALL_MODES:
    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=mode,
        leave=False,
    ):
        prepared = prepare_parent(row)
        if prepared is None:
            continue

        chembl_id, parent_smiles, parent_desc = prepared

        try:
            proposed = propose_structures(
                parent_smiles,
                mode=mode,
                max_candidates=N_CANDIDATES_PER_PARENT,
                random_seed=RANDOM_SEED,
            )
        except Exception:
            continue

        for proposal in proposed:
            append_candidate_record(
                records=records,
                chembl_id=chembl_id,
                parent_smiles=parent_smiles,
                generated_smiles=proposal["generated_smiles"],
                parent_desc=parent_desc,
                mode=mode,
            )

df_scaffold_tuner = pd.DataFrame(records)

## 4. Save generated products

The output columns intentionally match the Random substitution output so that all methods can be evaluated with the same downstream analysis code.

In [5]:
OUTPUT_FILE = "../results/scaffold_tuner.csv"

df_scaffold_tuner.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")

Saved: ../results/scaffold_tuner.csv


## Notes for downstream analysis

The output format is kept compatible with the common analysis notebook.

Because generation is mode-directed, the intended target is:

- `add_hbd` → `delta_hbd = +1`
- `remove_hbd` → `delta_hbd = -1`
- `add_hba` → `delta_hba = +1`
- `remove_hba` → `delta_hba = -1`
- `add_ar` → `delta_ar = +1`
- `remove_ar` → `delta_ar = -1`
- `add_rb` → `delta_rotb = +1`
- `remove_rb` → `delta_rotb = -1`

No internal scaffold-tuner implementation details are reimplemented in this notebook.
